In [4]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    if random.random() < 0.05: # 5% szansy na fraud
        # Podejrzana transakcja
        return {
            'tx_id': f'TX{random.randint(1000,9999)}',
            'user_id': f'u{random.randint(1,20):02d}',
            'amount': round(random.uniform(3000.0, 5000.0), 2),
            'store': random.choice(sklepy),
            'category': 'elektronika',
            'hour': random.randint(0, 5),
            'timestamp': datetime.now().isoformat(),
        }
    else:
        # Normalna transakcja
        return {
            'tx_id': f'TX{random.randint(1000,9999)}',
            'user_id': f'u{random.randint(1,20):02d}',
            'amount': round(random.uniform(5.0, 2999.0), 2),
            'store': random.choice(sklepy),
            'category': random.choice(kategorie),
            'hour': random.randint(6, 22),
            'timestamp': datetime.now().isoformat(),
        }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    #Kategoria i godzina do printa
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | Kat: {tx['category']:11} | Godzina: {tx['hour']:02d}")
    time.sleep(0.5)

producer.flush()
producer.close()

Overwriting producer.py


In [5]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='filter-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję... (amount > 1000)\n")

for message in consumer:
    tx = message.value
    # rozwiazanie
    if tx['amount'] > 1000:
        print(f"ALERT: {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")

Writing consumer_filter.py


In [6]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='enrich-group', # INNY group_id!
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

def get_risk_level(amount):
    # przypisanie poziomu ryzyka na podstawie kwoty
    if amount > 3000:
        return 'HIGH'
    if amount > 1000:
        return 'MEDIUM'
    return 'LOW'

print("Nasłuchuję i oceniam ryzyko...\n")

for message in consumer:
    tx = message.value
    # Dodanie nowego pola 'risk_level' do słownika transakcji
    tx['risk_level'] = get_risk_level(tx['amount'])
    
    # Wyświetlenie wyniku
    print(f"[{tx['risk_level']:6s}] {tx['tx_id']} | {tx['amount']:.2f} PLN")

Writing consumer_enrich.py


In [7]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter, defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()      # domyślna wartość: 0
total_amount = defaultdict(float) # domyślna wartość: 0.0
msg_count = 0

print("Nasłuchuję i zliczam transakcje...\n")

for message in consumer:
    tx = message.value
    store = tx['store']
    
    # 1. Zwiekszenie licznika transakcji dla danego sklepu
    store_counts[store] += 1
    
    # 2. Dodajemy kwotę transakcji do sumy danego sklepu
    total_amount[store] += tx['amount']
    
    msg_count += 1
    
    # 3. Co 10 wiadomości: wyświetlamy tabelę podsumowującą
    if msg_count % 10 == 0:
        print(f"\n{'='*52}")
        print(f"{'Sklep':<12} {'Liczba':>8} {'Suma':>12} {'Śr.':>10}")
        print(f"{'-'*52}")
        for s in sorted(store_counts):
            n = store_counts[s]
            tot = total_amount[s]
            print(f"{s:<12} {n:>8} {tot:>12.2f} {tot/n:>10.2f}")
        print(f"{'='*52}\nŁącznie: {msg_count} wiadomości\n")

Writing consumer_count.py


In [8]:
from datetime import datetime

def score_transaction(tx):
    score = 0
    rules = []
    
    # R1
    if tx['amount'] > 3000:
        score += 3
        rules.append('R1:high_amount')
        
    # R2: 
    if tx.get('category') == 'elektronika' and tx['amount'] > 1500:
        score += 2
        rules.append('R2:electronics_high')
        
    # R3: 
    hour = tx.get('hour')
    if hour is None:
        # wyciągnij z timestamp jeśli nie ma pola 'hour'
        try:
            hour = datetime.fromisoformat(tx['timestamp']).hour
        except Exception:
            hour = 12 # bezpieczna wartość domyślna
            
    if hour < 6:
        score += 2
        rules.append('R3:night')
        
    return score, rules

# Test działania funkcji 
test_tx = {
    'tx_id': 'TX999', 
    'amount': 4500.0, 
    'category': 'elektronika',
    'timestamp': '2026-04-01T03:15:00'
}

score, rules = score_transaction(test_tx)
print(f"Score: {score}, Reguły: {rules}")
print(f"Podejrzana: {score >= 3}")

Score: 7, Reguły: ['R1:high_amount', 'R2:electronics_high', 'R3:night']
Podejrzana: True


In [9]:
%%file scoring_consumer.py
from kafka import KafkaConsumer, KafkaProducer
from datetime import datetime
import json

consumer = KafkaConsumer(
    'transactions', 
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest', 
    group_id='scoring-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

alert_producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

# funkcja scoringowa
def score_transaction(tx):
    score = 0
    rules = []
    
    if tx['amount'] > 3000:
        score += 3; rules.append('R1')
    if tx.get('category') == 'elektronika' and tx['amount'] > 1500:
        score += 2; rules.append('R2')
        
    hour = tx.get('hour', datetime.fromisoformat(tx['timestamp']).hour)
    if hour < 6:
        score += 2; rules.append('R3')
        
    return score, rules

print("Konsument scoringowy uruchomiony...\n")

# rozwiazanie
for message in consumer:
    tx = message.value
    # 1. Scoruj
    score, rules = score_transaction(tx)
    
    # 2. Jeśli >= 3: wyślij do 'alerts' i wypisz ALERT
    if score >= 3:
        alert = {**tx, 'score': score, 'rules': rules, 'alert': True}
        alert_producer.send('alerts', value=alert)
        print(f"ALERT (score {score}) {tx['tx_id']} | {tx['amount']:.2f} PLN | reguły: {rules}")
        alert_producer.flush()

Writing scoring_consumer.py
